# Week 2 Text Cleaning Exploration

This notebook profiles the raw listing remarks, reviews the cleaned output, and compares before/after examples.

In [1]:
import re
import sys

import pandas as pd

---

## 1. Artifacts Loading

Load the Week 1 listing sample and the Week 2 cleaned dataset.

In [2]:
raw = pd.read_csv('../data/processed/listing_sample.csv')
cleaned = pd.read_csv('../data/processed/listing_sample_cleaned.csv')

In [3]:
raw.shape, cleaned.shape

((1000, 7), (1000, 8))

In [4]:
cleaned.head(3)

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,remarks_cleaned
0,1159550963,29667 Fortitude Drive,Menifee,5.0,4.0,752490,Move in just in time for summer! Our popular ...,move in just in time for summer our popular pr...
1,1155343150,11379 Reidy Canyon,Escondido,8.0,8.0,2825000,Rare multi-generational estate offering three ...,rare multi generational estate offering three ...
2,1158531238,2689 E Towhee,Ontario,4.0,3.0,855000,"Welcome to 2689 E Towhee St., an exquisite sin...","welcome to 2689 e towhee st., an exquisite sin..."


---

## 2. Raw Dataset Profile

Before judging the cleaner, we first understand what kinds of text issues the pipeline needs to handle.

### 2.1 Shape and Missingness

Start with the basic structure and missingness profile.

In [5]:
raw_profile = pd.DataFrame({
    'dtype': raw.dtypes,
    'missing': raw.isna().sum(),
    'missing_pct': raw.isna().mean().round(3),
})

raw_profile

,dtype,missing,missing_pct
L_ListingID,int64,0,0.000
L_Address,object,0,0.000
L_City,object,2,0.002
beds,float64,2,0.002
baths,float64,0,0.000
price,int64,0,0.000
remarks,object,0,0.000


In [6]:
raw['remarks'].str.len().describe().round(1)

count    1000.0
mean     1277.9
std       574.0
min        83.0
25%       874.2
50%      1227.5
75%      1587.0
max      3963.0
Name: remarks, dtype: float64

### 2.2 Raw Text Profile

Use the same cleaner object that powers the pipeline to summarize the raw remark field before looking at individual patterns.

In [7]:
sys.path.append('..')
from src.real_estate_nlp.text_cleaner import TextCleaner

cleaner = TextCleaner()
raw_text_profile = cleaner.profile_column(raw, 'remarks')

pd.Series({
    'null_rate': raw_text_profile['null_rate'],
    'avg_length': raw_text_profile['avg_length'],
    'price_mentions': raw_text_profile['price_mentions'],
    'has_html': raw_text_profile['has_html'],
}).to_frame('value')

,value
null_rate,0.000
avg_length,1277.932
price_mentions,79.000
has_html,0.000


In [8]:
pd.DataFrame(
    raw_text_profile['common_abbreviations'][:15],
    columns=['abbreviation', 'count']
)

,abbreviation,count
0,hoa,135
1,adu,120
2,hvac,54
3,w/,49
4,a/c,43
5,ac,39
6,fridge,26
7,mid,24
8,approx,21
9,hwy,18


### 2.3 Raw Normalization Signals

These patterns show where normalization can reduce surface variation, even when the raw remarks are not heavily corrupted.

In [9]:
raw_patterns = {
    'uppercase_tokens': r'\b[A-Z]{2,}\b',
    'hyphenated_phrases': r'\b\w+-\w+\b',
    'slash_phrases': r'w/|w/o|a/c|w/d',
    'price_mentions': r'\$\s*\d|\b\d+(?:\.\d+)?[km]\b',
    'number_with_commas': r'\b\d{1,3}(?:,\d{3})+\b',
    'square_footage_mentions': r'\b\d[\d,]*(?:\.\d+)?\s*(?:sq\.?\s*ft\.?|sqft|sf|s\.f\.|square\s*(?:feet|foot))\b',
    'bedroom_mentions': r'\b\d+(?:\.\d+)?\s*(?:br|bd|bdrm|bed|beds|bedroom|bedrooms)\b',
    'bathroom_mentions': r'\b\d+(?:\.\d+)?\s*(?:ba|bth|bath|baths|bathroom|bathrooms)\b',
    'compact_bed_bath': r'\b\d+\s*/\s*\d+(?:\.\d+)?\b',
    'acre_mentions': r'\b(?:\d+(?:\.\d+)?|\d+\s*/\s*\d+)\s*(?:ac|acre|acres)\b',
    'hoa_mentions': r'\bhoa\b|homeowners association',
    'garage_or_parking_mentions': r'\bgarage\b|\bparking\b|\bcarport\b|\bdriveway\b|\bpkg\b|\bprkg\b',
    'year_built_mentions': r'\bbuilt in \d{4}\b|\b\d{4}-built\b|\byr\s*blt\b',
    'story_or_level_mentions': r'\bsingle-story\b|\btwo-story\b|\b\d+-story\b|\btri-level\b|\bsplit-level\b',
    'html_or_entity': r'<[^>]+>|&(?:nbsp|amp|lt|gt);',
}

raw_signal_counts = pd.Series({
    name: raw['remarks'].str.contains(pattern, case=False, regex=True).sum()
    for name, pattern in raw_patterns.items()
}).sort_values(ascending=False)

raw_signal_counts.to_frame('raw_count')

,raw_count
uppercase_tokens,1000
hyphenated_phrases,887
garage_or_parking_mentions,595
number_with_commas,324
square_footage_mentions,308
bedroom_mentions,308
bathroom_mentions,297
hoa_mentions,125
story_or_level_mentions,111
price_mentions,79


---

## 3. Cleaned Dataset Profile

Now profile the cleaned artifact using the same lens. The main question is whether the obvious raw shorthand has been reduced and replaced by stable phrases.

### 3.1 Cleaned Field Profile

The cleaned dataset should keep the same rows and add a usable normalized text field.

In [10]:
cleaned_profile = pd.DataFrame({
    'dtype': cleaned.dtypes,
    'missing': cleaned.isna().sum(),
    'missing_pct': cleaned.isna().mean().round(3),
})

cleaned_profile

,dtype,missing,missing_pct
L_ListingID,int64,0,0.000
L_Address,object,0,0.000
L_City,object,2,0.002
beds,float64,2,0.002
baths,float64,0,0.000
price,int64,0,0.000
remarks,object,0,0.000
remarks_cleaned,object,0,0.000


In [11]:
length_profile = pd.DataFrame({
    'raw_chars': cleaned['remarks'].str.len(),
    'cleaned_chars': cleaned['remarks_cleaned'].str.len(),
})
length_profile['char_delta'] = length_profile['cleaned_chars'] - length_profile['raw_chars']

length_profile.describe().round(1)

,raw_chars,cleaned_chars,char_delta
count,1000.0,1000.0,1000.0
mean,1277.9,1280.5,2.6
std,574.0,573.4,21.8
min,83.0,82.0,-234.0
25%,874.2,883.8,-6.0
50%,1227.5,1227.5,-1.0
75%,1587.0,1590.2,12.0
max,3963.0,3962.0,109.0


Abbreviation expansion can make cleaned remarks slightly longer. Large changes are worth checking in the examples section.

### 3.2 Remaining Normalization Signals

Run the same signal checks on `remarks_cleaned` to see which forms were reduced and which still remain.

In [12]:
remaining_signal_counts = pd.Series({
    name: cleaned['remarks_cleaned'].str.contains(pattern, case=False, regex=True).sum()
    for name, pattern in raw_patterns.items()
}).sort_values(ascending=False)

pd.DataFrame({
    'raw_count': raw_signal_counts,
    'cleaned_count': remaining_signal_counts,
    'change': remaining_signal_counts - raw_signal_counts,
}).fillna(0).astype(int)

,raw_count,cleaned_count,change
acre_mentions,64,99,35
bathroom_mentions,297,493,196
bedroom_mentions,308,533,225
compact_bed_bath,40,0,-40
garage_or_parking_mentions,595,595,0
hoa_mentions,125,125,0
html_or_entity,0,0,0
hyphenated_phrases,887,0,-887
number_with_commas,324,0,-324
price_mentions,79,0,-79


### 3.3 Standardized Terms After Cleaning

These terms are useful for Week 3 extraction because they reduce the number of surface forms the extractor has to recognize.

In [13]:
standard_terms = [
    'bedroom',
    'bathroom',
    'square feet',
    'acre',
    'homeowners association',
    'parking',
    'garage',
    'year built',
    'stainless steel',
]

pd.Series({
    term: cleaned['remarks_cleaned'].str.contains(re.escape(term), case=False, regex=True).sum()
    for term in standard_terms
}).sort_values(ascending=False).to_frame('cleaned_count')

,cleaned_count
bedroom,838
bathroom,666
garage,458
square feet,317
parking,252
stainless steel,185
acre,149
homeowners association,125
year built,51


---

## 4. Before and After Examples

### 4.1 Largest Text Changes

These examples show the strongest effect of abbreviation expansion and punctuation cleanup.

In [14]:
review = cleaned[['remarks', 'remarks_cleaned']].copy()
review['char_delta'] = review['remarks_cleaned'].str.len() - review['remarks'].str.len()

review.sort_values('char_delta', key=lambda s: s.abs(), ascending=False)[
    ['remarks', 'remarks_cleaned']
].head(8)

,remarks,remarks_cleaned
110,"Turnkey, four bedroom Redlands home with RV pa...","turnkey, four bedroom redlands home with rv pa..."
95,Price Improvement and Seller may pay up to $20...,price improvement and seller may pay up to 200...
687,A stunning to-the-studs transformation in the ...,a stunning to the studs transformation in the ...
826,Turn-Key Perfection in North Escondido! Move i...,turn key perfection in north escondido move in...
396,Welcome to this stunning French Country-style ...,welcome to this stunning french country style ...
895,1380 Oak Hill in Escondido HOME For Sale! Rare...,1380 oak hill in escondido home for sale rare ...
596,Permits are near approval to convert the exist...,permits are near approval to convert the exist...
470,Massive Price Improvement!! This expansive Chi...,massive price improvement. this expansive chic...


### 4.2 Representative Sample

A fixed sample gives a less extreme view of how the cleaner behaves across ordinary rows.

In [15]:
cleaned.loc[
    cleaned['remarks'] != cleaned['remarks_cleaned'],
    ['remarks', 'remarks_cleaned']
].sample(10, random_state=0)

,remarks,remarks_cleaned
993,Two homes on one lot! 3 bed + 2 bath Main dwel...,two homes on one lot 3 bedroom 2 bathroom main...
859,Dual use zoning makes this property ideal for ...,dual use zoning makes this property ideal for ...
298,Welcome to this charming two-story townhouse i...,welcome to this charming 2 story townhouse in ...
553,"Serene Storybook Estate with Old World Charm, ...","serene storybook estate with old world charm, ..."
672,A rare opportunity to acquire one of Cheviot H...,a rare opportunity to acquire one of cheviot h...
971,Welcome to your one-level View Heights dream h...,welcome to your 1 story view heights dream hom...
27,"Set on 2.30 private acres in Yucca Valley, thi...","set on 2.30 private acres in yucca valley, thi..."
231,Top floor. Vaulted ceilings. Move-in ready. Th...,top floor. vaulted ceilings. move in ready. th...
306,Welcome to this spacious Riverside HORSE PROPE...,welcome to this spacious riverside horse prope...
706,This is a Plan 3 Lark home by Tri Pointe Homes...,this is a plan 3 lark home by tri pointe homes...
